# miniF2F Lean 4 Evaluation v2

**修复内容**:
- 移除不存在的 `lean --make` 选项
- 正确构建 Mathlib 依赖
- 使用 `lake build` 验证 proof

In [1]:
# ================================================================
#  Cell 1: 挂载 Drive & 安装依赖
# ================================================================
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate peft datasets torch sentencepiece protobuf

import torch


Mounted at /content/drive


In [2]:
# ================================================================
#  Cell 2: 安装 Lean 4 & 构建 Mathlib（关键步骤！）
# ================================================================
import subprocess, os

elan_path = os.path.expanduser("~/.elan/bin")
os.environ["PATH"] = f"{elan_path}:{os.environ.get('PATH', '')}"
os.environ["ELAN_HOME"] = os.path.expanduser("~/.elan")

# --- 安装 elan ---
if not os.path.exists(f"{elan_path}/lean"):
    print("=== Installing elan ===")
    subprocess.run(
        'curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y',
        shell=True, timeout=300
    )
else:
    print("elan already installed")

# --- 克隆 miniF2F 项目 ---
MINIF2F_DIR = "/content/minif2f-lean4"
if os.path.exists(MINIF2F_DIR):
    import shutil
    shutil.rmtree(MINIF2F_DIR)
    print("Cleaned old miniF2F dir")

print("=== Cloning miniF2F-lean4 ===")
subprocess.run(
    f"git clone --depth 1 https://github.com/rahul3613/miniF2F-lean4 {MINIF2F_DIR}",
    shell=True, timeout=120, check=True
)

# 查看 lean-toolchain
with open(f"{MINIF2F_DIR}/lean-toolchain") as f:
    tc = f.read().strip()
print(f"Lean toolchain: {tc}")

# --- 构建 Mathlib（耗时较长，约 10-20 分钟）---
print("\n=== Downloading Mathlib prebuilt cache ===")
r = subprocess.run(
    "lake exe cache get",
    shell=True, cwd=MINIF2F_DIR, capture_output=True, text=True, timeout=1800
)
if r.returncode == 0:
    print("Mathlib cache downloaded successfully!")
else:
    print(f"Cache get exit code: {r.returncode}")
    print(f"stdout: {r.stdout[-300:]}")
    print(f"stderr: {r.stderr[-300:]}")
    print("Will try building from source...")

print("\n=== Building Mathlib (this will take 10-30 min) ===")
print("Please wait... do NOT interrupt.")
r = subprocess.run(
    "lake build",
    shell=True, cwd=MINIF2F_DIR, capture_output=True, text=True, timeout=7200
)
if r.returncode == 0:
    print("\nMathlib build SUCCESS!")
else:
    print(f"\nMathlib build exit code: {r.returncode}")
    print(f"stdout: {r.stdout[-500:]}")
    print(f"stderr: {r.stderr[-500:]}")

# --- 验证 Lean 环境是否正常 ---
print("\n=== Quick sanity check: lean --version ===")
r = subprocess.run(
    ["lean", "--version"],
    capture_output=True, text=True, cwd=MINIF2F_DIR
)
print(r.stdout.strip())

# 测试: 一个最简单的 Mathlib import 能不能通过
print("\n=== Test: Mathlib.Data.Real.Basic import ===")
os.makedirs(f"{MINIF2F_DIR}/MiniF2FLean4", exist_ok=True)
test_file = f"{MINIF2F_DIR}/MiniF2FLean4/TestImport.lean"
with open(test_file, "w") as f:
    f.write("import Mathlib.Data.Real.Basic\n#eval (1 : Real) + 1\n")

r = subprocess.run(["lake", "env", "lean", test_file], capture_output=True, text=True, cwd=MINIF2F_DIR, timeout=120)
print(f"exit code: {r.returncode}")
if r.returncode != 0:
    print(f"stdout: {r.stdout[:300]}")
    print(f"stderr: {r.stderr[:300]}")
    print("\nERROR: Mathlib import still failing. Stopping.")
    print("Please send this output for debugging.")
else:
    print(f"stdout: {r.stdout.strip()}")
    print("\nLean 4 + Mathlib is working correctly!")

# 清理
if os.path.exists(test_file):
    os.remove(test_file)
for ext in ['.olean', '.ilean']:
    f = test_file.replace('.lean', ext)
    if os.path.exists(f):
        os.remove(f)

LEAN_READY = r.returncode == 0
print(f"\nLEAN_READY = {LEAN_READY}")

=== Installing elan ===
=== Cloning miniF2F-lean4 ===
Lean toolchain: leanprover/lean4:v4.6.0

=== Downloading Mathlib prebuilt cache ===
Mathlib cache downloaded successfully!

=== Building Mathlib (this will take 10-30 min) ===
Please wait... do NOT interrupt.

Mathlib build SUCCESS!

=== Quick sanity check: lean --version ===
Lean (version 4.6.0, x86_64-unknown-linux-gnu, commit a5bc9013ab13, Release)

=== Test: Mathlib.Data.Real.Basic import ===
exit code: 0
stdout: Real.ofCauchy (sorry /- 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ... -/)

Lean 4 + Mathlib is working correctly!

LEAN_READY = True


In [3]:
# ================================================================
#  Cell 3: 加载数据集
# ================================================================
from datasets import load_dataset
import json

ds = load_dataset("cat-searcher/minif2f-lean4")
valid_ds = ds["validation"]
test_ds = ds["test"]
print(f"Validation: {len(valid_ds)}, Test: {len(test_ds)}")
print(f"Columns: {valid_ds.column_names}")

# 检查统一 header
headers = set(ex['header'] for ex in valid_ds)
print(f"Unique headers: {len(headers)}")
UNIVERSAL_HEADER = valid_ds[0]['header']
print(f"Header length: {len(UNIVERSAL_HEADER)} chars")

README.md:   0%|          | 0.00/212 [00:00<?, ?B/s]

valid.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Validation: 244, Test: 244
Columns: ['id', 'split', 'formal_statement', 'header', 'informal_stmt', 'informal_proof']
Unique headers: 1
Header length: 385 chars


In [9]:
# ================================================================
#  Cell 4: 核心函数（lake env lean 修复版）
# ================================================================
import torch, gc, re, time, subprocess, os, uuid
from tqdm.auto import tqdm
from typing import List, Dict, Optional, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from math import comb

LEAN_LIB_DIR = os.path.join(MINIF2F_DIR, "MiniF2FLean4")
os.makedirs(LEAN_LIB_DIR, exist_ok=True)

def load_and_merge_model(base_model_name, lora_path=None):
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True, padding_side='left')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto")
    if lora_path and os.path.exists(lora_path):
        model = PeftModel.from_pretrained(model, lora_path).merge_and_unload()
    model.eval()
    return model, tokenizer

def build_prompt(formal_statement):
    return f"""<|im_start|>system
You are an expert Lean 4 theorem prover. Given a theorem ending with ":= sorry", replace sorry with a correct Lean 4 proof.
Output ONLY the tactic proof after "by". Do NOT repeat the theorem.<|im_end|>
<|im_start|>user
Prove this theorem in Lean 4:
{formal_statement}<|im_end|>
<|im_start|>assistant
"""

def extract_proof(text):
    text = re.sub(r'<\|im_end\|>', '', text).strip()
    # FIX: 清理所有残余的特殊 token（包括截断的 <|endo 等）
    text = re.sub(r'<\|[^>]*>?', '', text).strip()
    text = re.sub(r'<\|', '', text).strip()
    m = re.search(r':=\s*by\s*(.*)', text, re.DOTALL)
    if m:
        proof = m.group(1).strip()
        proof = re.sub(r'<\|[^>]*>?', '', proof)  # 再清一次
        proof = re.sub(r'<\|', '', proof)
        return proof
    m = re.search(r':=\s*(.*)', text, re.DOTALL)
    if m:
        p = re.sub(r'^theorem\s+.*?:=', '', m.group(1), flags=re.DOTALL).strip()
        p = re.sub(r'<\|[^>]*>?', '', p)
        p = re.sub(r'<\|', '', p)
        if p:
            return p
    # 去掉开头的 by
    text = re.sub(r'^by\s*', '', text)
    return text

def generate_proofs(model, tokenizer, problems, num_samples=8, temperature=0.6, top_p=0.95, max_new_tokens=512, batch_size=4):
    results = {p['id']: [] for p in problems}
    prompts, indices = [], []
    for i, prob in enumerate(problems):
        prompt = build_prompt(prob['formal_statement'])
        prompts.extend([prompt] * num_samples)
        indices.extend([i] * num_samples)
    print(f"Total: {len(prompts)} ({len(problems)} x {num_samples})")
    outputs = []
    for start in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch = prompts[start:start + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p, do_sample=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        ilen = inputs['input_ids'].shape[1]
        outputs.extend(tokenizer.decode(g[ilen:], skip_special_tokens=False) for g in gen)
        if start % (batch_size * 10) == 0:
            torch.cuda.empty_cache()
    for out, idx in zip(outputs, indices):
        results[problems[idx]['id']].append(extract_proof(out))
    return results

def verify_proof_with_lean(header, formal_statement, proof, timeout=300):
    theorem = re.sub(r':=\s*(?:by\s*)?sorry\s*$', '', formal_statement.rstrip())
    full = f"{theorem} := by\n  {proof.strip()}" if proof.strip() else f"{theorem} := sorry"
    content = f"{header}\n\n{full}\n"
    fname = f"Eval_{uuid.uuid4().hex[:8]}.lean"
    fpath = os.path.join(LEAN_LIB_DIR, fname)
    try:
        with open(fpath, 'w') as f:
            f.write(content)
        env = os.environ.copy()
        env["PATH"] = f"{elan_path}:{env.get('PATH', '')}"
        env["ELAN_HOME"] = os.path.expanduser("~/.elan")
        r = subprocess.run(["lake", "env", "lean", fpath], capture_output=True, text=True, timeout=timeout, cwd=MINIF2F_DIR, env=env)
        success = r.returncode == 0
        err = (r.stdout or "")[:500]
        if not success and r.stderr:
            err += " | " + r.stderr[:300]
        return success, err
    except subprocess.TimeoutExpired:
        return False, f"Timeout ({timeout}s)"
    except Exception as e:
        return False, str(e)
    finally:
        for p in [fpath]:
            if os.path.exists(p):
                try: os.remove(p)
                except: pass
        for ext in ['.olean', '.ilean']:
            f = fpath.replace('.lean', ext)
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1.0 - comb(n - c, k) / comb(n, k)

def compute_pass_at_k(all_results, k_values):
    scores = {k: [] for k in k_values}
    for pid, res in all_results.items():
        n, c = len(res), sum(res)
        for k in k_values:
            scores[k].append(pass_at_k(n, c, k))
    return {k: sum(scores[k]) / len(scores[k]) for k in k_values}

print("Functions ready")

Functions ready


In [10]:
# ================================================================
#  Cell 5: 模型配置（路径保持不变）
# ================================================================
import os

OUTPUT_DIR = "/content/drive/MyDrive/AIMS5790/eval_results_minif2f"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_CONFIGS = [
    {
        "name": "Qwen2.5-0.5B + QLoRA + baby_step_s42",
        "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
        "lora_path": "/content/drive/MyDrive/AIMS5790/outputs/curriculum/baby_step_s42/adapter",
    },
    {
        "name": "Qwen2.5-0.5B + QLoRA + curriculum_abs_influence_asc",
        "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
        "lora_path": "/content/drive/MyDrive/AIMS5790/outputs/experiments_p2/curriculum_abs_influence_asc/adapter",
    },
    {
        "name": "Qwen2.5-0.5B + QLoRA + curriculum_abs_influence_desc",
        "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
        "lora_path": "/content/drive/MyDrive/AIMS5790/outputs/experiments_p2/curriculum_abs_influence_desc/adapter",
    },
    {
        "name": "Qwen2.5-0.5B + QLoRA + curriculum_sandwich_influence",
        "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
        "lora_path": "/content/drive/MyDrive/AIMS5790/outputs/experiments_p2/curriculum_sandwich_influence/adapter",
    },
    {
        "name": "Qwen2.5-0.5B + QLoRA + curriculum_hybrid_influence_diff",
        "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
        "lora_path": "/content/drive/MyDrive/AIMS5790/outputs/experiments_p2/curriculum_hybrid_influence_diff/adapter",
    },
    {
        "name": "Qwen2.5-0.5B + QLoRA + curriculum_importance_sampling",
        "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
        "lora_path": "/content/drive/MyDrive/AIMS5790/outputs/experiments_p2/curriculum_importance_sampling/adapter",
    },
]

# === 评测参数 ===
EVAL_SPLIT = "validation"
NUM_SAMPLES = 8
TEMPERATURE = 0.6
TOP_P = 0.95
MAX_NEW_TOKENS = 512
BATCH_SIZE = 4
VERIFY_TIMEOUT = 300
K_VALUES = [1, 2, 4, 8]
EVAL_LIMIT = 30   # 0 = 全部; 小数字如 10 = 快速测试

print("Config ready")
print(f"Models: {len(MODEL_CONFIGS)}")
print(f"Lean ready: {LEAN_READY}")
for c in MODEL_CONFIGS:
    lora = f" + LoRA" if c['lora_path'] else ""
    print(f"  - {c['name']}{lora}")

Config ready
Models: 6
Lean ready: True
  - Qwen2.5-0.5B + QLoRA + baby_step_s42 + LoRA
  - Qwen2.5-0.5B + QLoRA + curriculum_abs_influence_asc + LoRA
  - Qwen2.5-0.5B + QLoRA + curriculum_abs_influence_desc + LoRA
  - Qwen2.5-0.5B + QLoRA + curriculum_sandwich_influence + LoRA
  - Qwen2.5-0.5B + QLoRA + curriculum_hybrid_influence_diff + LoRA
  - Qwen2.5-0.5B + QLoRA + curriculum_importance_sampling + LoRA


In [11]:
# ================================================================
#  Cell 6: 运行完整评测
# ================================================================
import json, time, gc
from datetime import datetime

ds = load_dataset("cat-searcher/minif2f-lean4")
eval_ds = ds[EVAL_SPLIT]

problems = []
for ex in eval_ds:
    problems.append({
        'id': ex['id'],
        'formal_statement': ex['formal_statement'],
        'header': ex['header'],
    })
if EVAL_LIMIT > 0:
    problems = problems[:EVAL_LIMIT]

print(f"Evaluating {len(problems)} problems from {EVAL_SPLIT}")
if not LEAN_READY:
    print("WARNING: Lean 4 not ready! Using fallback heuristic.")

all_model_results = {}
comparison_table = []

for model_cfg in MODEL_CONFIGS:
    name = model_cfg['name']
    print(f"\n{'='*70}")
    print(f"  {name}")
    print(f"{'='*70}")
    t0 = time.time()

    # Load model
    try:
        model, tokenizer = load_and_merge_model(model_cfg['base_model'], model_cfg.get('lora_path'))
    except Exception as e:
        print(f"ERROR: {e}")
        continue

    # Generate
    t1 = time.time()
    proofs = generate_proofs(model, tokenizer, problems, NUM_SAMPLES, TEMPERATURE, TOP_P, MAX_NEW_TOKENS, BATCH_SIZE)
    print(f"Generation: {(time.time()-t1)/60:.1f} min")

    # Verify
    print("Verifying proofs...")
    verification_results = {}
    verification_details = {}

    if LEAN_READY:
        total = len(problems) * NUM_SAMPLES
        ok = 0
        for prob in tqdm(problems, desc="Verifying"):
            pid = prob['id']
            verification_results[pid] = []
            verification_details[pid] = []
            for proof_text in proofs.get(pid, []):
                success, err = verify_proof_with_lean(prob['header'], prob['formal_statement'], proof_text, VERIFY_TIMEOUT)
                verification_results[pid].append(success)
                verification_details[pid].append({'proof': proof_text, 'success': success, 'error': err})
                if success:
                    ok += 1
        print(f"Lean verification: {ok}/{total} accepted")
    else:
        for prob in problems:
            pid = prob['id']
            verification_results[pid] = []
            verification_details[pid] = []
            for proof_text in proofs.get(pid, []):
                valid = len(proof_text.strip()) > 10 and 'sorry' not in proof_text.lower()
                verification_results[pid].append(valid)
                verification_details[pid].append({'proof': proof_text, 'success': valid, 'error': 'heuristic'})

    # Compute pass@k
    scores = compute_pass_at_k(verification_results, K_VALUES)
    elapsed = time.time() - t0

    print(f"\n--- {name} Results ---")
    for k in K_VALUES:
        print(f"  pass@{k}: {scores[k]*100:.2f}%")
    print(f"  Time: {elapsed/60:.1f} min")

    # Save
    result = {
        'model_name': name, 'config': model_cfg, 'split': EVAL_SPLIT,
        'num_problems': len(problems), 'num_samples': NUM_SAMPLES,
        'pass_at_k': {f'pass@{k}': f"{v*100:.2f}%" for k, v in scores.items()},
        'verification': 'lean4' if LEAN_READY else 'heuristic',
        'timestamp': datetime.now().isoformat(),
        'per_problem': {}
    }
    for prob in problems:
        pid = prob['id']
        result['per_problem'][pid] = {
            'formal_statement': prob['formal_statement'],
            'correct': sum(verification_results.get(pid, [])),
            'proofs': verification_details.get(pid, []),
        }

    safe_name = name.replace('/', '_').replace(' ', '_')
    path = os.path.join(OUTPUT_DIR, f"{safe_name}_v2.json")
    with open(path, 'w') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"Saved: {path}")

    all_model_results[name] = result
    comparison_table.append({
        'model': name,
        **{f'pass@{k}': f"{scores[k]*100:.2f}%" for k in K_VALUES},
        'time_min': f"{elapsed/60:.1f}",
        'verified': LEAN_READY,
    })

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# Save comparison
comp_path = os.path.join(OUTPUT_DIR, "comparison_v2.json")
with open(comp_path, 'w') as f:
    json.dump({'models': comparison_table, 'timestamp': datetime.now().isoformat()}, f, indent=2)

print(f"\n{'='*70}")
print("DONE!")
print(f"{'='*70}")

Evaluating 30 problems from validation

  Qwen2.5-0.5B + QLoRA + baby_step_s42


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Total: 240 (30 x 8)


Generating:   0%|          | 0/60 [00:00<?, ?it/s]

Generation: 2.7 min
Verifying proofs...


...

Total: 240 (30 x 8)


Generating:   0%|          | 0/60 [00:00<?, ?it/s]

Generation: 6.1 min
Verifying proofs...


Verifying:   0%|          | 0/30 [00:00<?, ?it/s]

Lean verification: 5/240 accepted

--- Qwen2.5-0.5B + QLoRA + curriculum_importance_sampling Results ---
  pass@1: 2.08%
  pass@2: 3.45%
  pass@4: 4.95%
  pass@8: 6.67%
  Time: 12.3 min
Saved: /content/drive/MyDrive/AIMS5790/eval_results_minif2f/Qwen2.5-0.5B_+_QLoRA_+_curriculum_importance_sampling_v2.json

DONE!


In [8]:
# ================================================================
#  调试: 看看 Lean 为什么拒绝 proof
# ================================================================
import json, os, subprocess

result_path = "/content/drive/MyDrive/AIMS5790/eval_results_minif2f/Qwen2.5-0.5B_+_QLoRA_+_baby_step_s42_v2.json"

with open(result_path, "r") as f:
    result = json.load(f)

elan_path = os.path.expanduser("~/.elan/bin")
MINIF2F_DIR = "/content/minif2f-lean4"
env = os.environ.copy()
env["PATH"] = f"{elan_path}:{env.get('PATH', '')}"
env["ELAN_HOME"] = os.path.expanduser("~/.elan")

# 取前3个问题，每个看1个proof的完整Lean错误
count = 0
for pid, data in result['per_problem'].items():
    if count >= 3:
        break
    print(f"\n{'='*70}")
    print(f"Problem: {pid}")
    print(f"Statement: {data['formal_statement'][:200]}")

    for i, pinfo in enumerate(data['proofs'][:1]):
        proof = pinfo['proof']
        print(f"\n--- Generated proof ({len(proof)} chars) ---")
        print(proof[:800])

        # 手动跑一遍lean看完整错误
        header = result.get('per_problem', {}).get(pid, {}).get('formal_statement', '')
        # 从dataset拿header
        from datasets import load_dataset
        ds = load_dataset("cat-searcher/minif2f-lean4")["validation"]
        ex = next(e for e in ds if e['id'] == pid)
        header = ex['header']

        import re, uuid
        theorem = re.sub(r':=\s*(?:by\s*)?sorry\s*$', '', data['formal_statement'].rstrip())
        full = f"{theorem} := by\n  {proof.strip()}" if proof.strip() else f"{theorem} := sorry"
        content = f"{header}\n\n{full}\n"

        LEAN_LIB_DIR = os.path.join(MINIF2F_DIR, "MiniF2FLean4")
        os.makedirs(LEAN_LIB_DIR, exist_ok=True)
        fpath = os.path.join(LEAN_LIB_DIR, f"debug_{pid}.lean")
        with open(fpath, 'w') as f:
            f.write(content)

        r = subprocess.run(["lake", "env", "lean", fpath], capture_output=True, text=True, timeout=120, cwd=MINIF2F_DIR, env=env)

        print(f"\n--- Lean exit code: {r.returncode} ---")
        print(f"stdout:\n{r.stdout[:1500]}")
        if r.stderr:
            print(f"stderr:\n{r.stderr[:500]}")

        # 清理
        for p in [fpath]:
            if os.path.exists(p):
                os.remove(p)

    count += 1


Problem: amc12a_2019_p21
Statement: theorem amc12a_2019_p21
  (z : ℂ)
  (h₀ : z = (1 + Complex.I) / Real.sqrt 2) :
  (∑ k in Finset.Icc 1 12, (z^(k^2))) * (∑ k in Finset.Icc 1 12, (1 / z^(k^2))) = 36 := sorry

--- Generated proof (1592 chars) ---
{simp}<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|end

In [ ]:
# ================================================================
#  Cell 7: 显示结果
# ================================================================
import pandas as pd

if comparison_table:
    df = pd.DataFrame(comparison_table)
    cols = ['model'] + [f'pass@{k}' for k in K_VALUES] + ['time_min']
    cols = [c for c in cols if c in df.columns]

    print(f"\n{'='*70}")
    print(f"  miniF2F Results ({EVAL_SPLIT}, {len(problems)} problems, {NUM_SAMPLES} samples)")
    print(f"  Verification: {'Lean 4' if LEAN_READY else 'Heuristic (NOT real)'}")
    print(f"{'='*70}\n")
    print(df[cols].to_string(index=False))

    for k in K_VALUES:
        col = f'pass@{k}'
        if col in df.columns:
            best = df[col].str.rstrip('%').astype(float).idxmax()
            print(f"Best pass@{k}: {df.loc[best,'model']} ({df.loc[best,col]})")

    # 保存 CSV
    csv_path = os.path.join(OUTPUT_DIR, "results_v2.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nCSV: {csv_path}")
else:
    print("No results yet. Run Cell 6 first.")


  miniF2F Results (validation, 30 problems, 8 samples)
  Verification: Lean 4

                                            model pass@1 pass@2 pass@4 pass@8 time_min
          Qwen2.5-0.5B + QLoRA + mixed_curriculum  0.83%  1.67%  3.33%  6.67%     13.8
           Qwen2.5-0.5B + QLoRA + filter_harmdata  1.25%  2.38%  4.29%  6.67%      7.5
    Qwen2.5-0.5B + QLoRA + prune_threshold_-10000  0.42%  0.83%  1.67%  3.33%     12.1
  Qwen2.5-0.5B + QLoRA + prune_remove_bottom10pct  0.42%  0.83%  1.67%  3.33%     11.6
      Qwen2.5-0.5B + QLoRA + prune_loss>0.9_inf<0  0.83%  1.55%  2.62%  3.33%     12.8
Qwen2.5-0.5B + QLoRA + prune_combined_remove10pct  0.42%  0.83%  1.67%  3.33%     11.7
            Qwen2.5-0.5B + QLoRA + influence_desc  0.83%  1.55%  2.62%  3.33%      9.3
             Qwen2.5-0.5B + QLoRA + influence_asc  2.08%  3.45%  4.95%  6.67%     12.1
           Qwen2.5-0.5B + QLoRA + anti_curriculum  0.83%  1.55%  2.62%  3.33%     11.7
Best pass@1: Qwen2.5-0.5B + QLoRA + influence_asc 